# 04_possession_model — Ball Possession Detection (More Than A Hack 2027)

Notebook tunggal yang menjalankan seluruh pipeline: **data mentah → 26 fitur → weak label → LightGBM → HMM + Viterbi → post-processing → `our_submission.csv` → validasi**.

## Cara menjalankan
1. Simpan file ini di folder `starter/` (atau di root `more-than-a-hack-2027/`). Sel pertama akan mencari folder root otomatis.
2. Pastikan data sudah ada di `data/tactical/` dan paket terpasang: `pip install -r requirements-starter.txt lightgbm scikit-learn matplotlib`.
3. Pilih **Run All**. Sel 1.1 paling lama (kloppy membaca 199.319 frame x 46 slot; bisa beberapa menit dan butuh RAM beberapa GB). Sel lain berjalan dalam detik sampai menit.
4. Hasil: `evaluation/results/our_submission.csv`, lalu sel 7 menjalankan `validate_submission.py` dan `run_evaluation.py`.

## Istilah singkat
- **Possession**: tim mana yang sedang menguasai bola (H = Home, A = Away, D = Dead ball, X = abstain).
- **spellF1**: metrik utama. Menilai keutuhan satu segmen possession, bukan akurasi per frame. Label yang sering berganti (*flickering*) merusak skor.
- **Weak label**: label murah hasil aturan (bukan ground truth) untuk melatih model.
- **Emission probability**: peluang P(H), P(A), P(D) per frame dari LightGBM.
- **HMM + Viterbi**: HMM (Hidden Markov Model) memodelkan peluang pindah state; Viterbi mencari urutan label terbaik untuk seluruh babak sekaligus, jadi label otomatis lebih mulus.
- **Forward-fill**: label X diisi evaluator dengan label valid terakhir.

## Keputusan desain yang perlu diketahui
- Fitur yang diimplementasikan berjumlah **26** (A:6, B:6, C:7, D:4, E:3), walau judul rancangan menulis 20.
- Nama tim **tidak di-hardcode**. Tag `Team` di Smart Tagging dipetakan ke Home/Away lewat kemiripan nama dengan metadata ATD, lalu diverifikasi dengan posisi pemain terdekat ke bola.
- Interval `SET PIECES` bertipe *Kick Off (Start)* dipotong sampai frame kickoff, karena jendelanya mencakup ±5 detik di sekitar kickoff dan tidak boleh melabeli `D` frame setelah bola mulai dimainkan.
- Public ground truth (5 menit) **hanya** untuk sanity check dan log-loss. Tidak ada tuning dari data itu.

## Section 0 — Setup & Import
Sel 0.1 mendeteksi root proyek otomatis, lalu memuat helper repo (`games.py`, `starter/load_atd.py`, `starter/load_smart_tagging.py`).

In [ ]:
import os
import sys
import re
import glob
import gc
import subprocess
import warnings
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import lightgbm as lgb
from sklearn.metrics import log_loss, f1_score

warnings.filterwarnings("ignore", category=RuntimeWarning)


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "games.py").is_file() and (folder / "starter").is_dir():
            return folder
        child = folder / "more-than-a-hack-2027"
        if (child / "games.py").is_file():
            return child
    raise FileNotFoundError(
        "Folder root proyek (berisi games.py dan starter/) tidak ditemukan. "
        "Jalankan notebook dari more-than-a-hack-2027/ atau subfolder starter/."
    )


ROOT = find_project_root()
os.chdir(ROOT)
for p in (str(ROOT), str(ROOT / "starter"), str(ROOT / "evaluation")):
    if p not in sys.path:
        sys.path.insert(0, p)

import games
import load_atd
import load_smart_tagging

RESULTS_DIR = ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT        :", ROOT)
print("DATA_DIR    :", games.DATA_DIR)

In [ ]:
GAME = games.GAME
FPS = games.GAMES[GAME]["fps"]
N_FRAMES = games.GAMES[GAME]["n_video_frames"]
PITCH_LENGTH, PITCH_WIDTH = games.GAMES[GAME]["pitch"]

PRE_MATCH = (0, 11184)
HALF1 = (11185, 90651)
HALFTIME = (90652, 110807)
HALF2_PRE = (110808, 113741)
HALF2 = (113742, 190197)
POST_MATCH = (190198, 199318)
PUBLIC_VAL_RANGE = (11185, 18684)

H_ID, A_ID, D_ID, X_ID = 0, 1, 2, -1
LABEL_TO_ID = {"H": H_ID, "A": A_ID, "D": D_ID, "X": X_ID}
ID_TO_LABEL = np.array(["H", "A", "D", "X"])

MIN_SPELL_FRAMES = 15
BALL_PROXIMITY_RADIUS = 3.0
TRANSITION_H_STAY = 0.9940
TRANSITION_D_STAY = 0.9700
EMISSION_WEIGHT_BALL = 0.7
EMISSION_WEIGHT_NO_BALL = 0.3
SET_PIECE_BUFFER_FRAMES = 5
BALL_NAN_FORWARD_FILL = 10
POST_SET_PIECE_DEAD_FRAMES = 25
ABSTAIN_MIN_BALL_GAP = 75
KICKOFF_PRIOR = 0.80
APPLY_ABSTAIN_RULE = True
BOOTSTRAP_RUNS = 1000

FEATURE_GROUPS = {
    "A": ["dist_home_min", "dist_away_min", "dist_ratio",
          "n_home_near_ball", "n_away_near_ball", "ball_zone_density_ratio"],
    "B": ["ball_x_m", "ball_y_m", "ball_in_home_half",
          "dist_ball_to_sideline", "dist_ball_to_endline", "ball_speed"],
    "C": ["centroid_home_x", "centroid_away_x", "centroid_diff_x",
          "spread_home", "spread_away",
          "dist_centroid_home_to_ball", "dist_centroid_away_to_ball"],
    "D": ["is_set_piece", "is_dead_ball_phase", "smart_tag_team", "frames_since_set_piece"],
    "E": ["frames_since_ball_seen", "ball_visible", "dist_ratio_roll25"],
}
FEATURES = [name for group in FEATURE_GROUPS.values() for name in group]
assert len(FEATURES) == 26, len(FEATURES)

graded_mask = np.zeros(N_FRAMES, dtype=bool)
for lo, hi in (HALF1, HALF2_PRE, HALF2):
    graded_mask[lo:hi + 1] = True


def intervals_to_mask(intervals):
    mask = np.zeros(N_FRAMES, dtype=bool)
    for start, end in intervals:
        start, end = max(0, int(start)), min(N_FRAMES, int(end))
        if end > start:
            mask[start:end] = True
    return mask


def tag_of(event, group):
    return (event["tags"].get(group) or "").strip().lower()


def code_of(event):
    return (event["code"] or "").strip().lower()


print("fitur:", len(FEATURES), "| frame graded:", int(graded_mask.sum()))

## Section 1 — Data Loading (ATD + Smart Tagging)
ATD sudah satu record per video frame; hanya perlu `video_frame = frame_id - 1`. Pemain dipisah menjadi slot Home/Away dari metadata kloppy (`team.ground`), bukan dari nama kolom.
Koordinat dikonversi ke meter (pitch 105,3 x 68,0 m) jika kloppy mengembalikan skala 0–1.

In [ ]:
alignment = load_atd.check_alignment_raw()
print(alignment)
assert alignment["agrees"], "ATD tidak selaras dengan jumlah frame video."

dataset = load_atd.load()
try:
    raw_df = dataset.to_df(engine="pandas")
except TypeError:
    raw_df = dataset.to_df()

raw_df["frame"] = raw_df["frame_id"].astype(int) - 1
raw_df = raw_df.set_index("frame")
raw_df = raw_df[~raw_df.index.duplicated()].reindex(np.arange(N_FRAMES))

slots, team_names = {"home": [], "away": []}, {}
for team in dataset.metadata.teams:
    side = "home" if "home" in str(team.ground).lower() else "away"
    team_names[side] = str(team.name)
    slots[side] = [p.player_id for p in team.players if f"{p.player_id}_x" in raw_df.columns]
assert slots["home"] and slots["away"], "Slot pemain Home/Away tidak ditemukan di kolom ATD."


def to_array(columns):
    return raw_df[columns].to_numpy(dtype=np.float64)


tracking = {
    "hx": to_array([f"{s}_x" for s in slots["home"]]),
    "hy": to_array([f"{s}_y" for s in slots["home"]]),
    "ax": to_array([f"{s}_x" for s in slots["away"]]),
    "ay": to_array([f"{s}_y" for s in slots["away"]]),
    "bx": raw_df["ball_x"].to_numpy(dtype=np.float64),
    "by": raw_df["ball_y"].to_numpy(dtype=np.float64),
    "bspeed": raw_df["ball_speed"].to_numpy(dtype=np.float64) if "ball_speed" in raw_df.columns else None,
}

if np.nanmax(np.abs(tracking["bx"])) <= 1.5:
    for key in ("hx", "ax", "bx"):
        tracking[key] = tracking[key] * PITCH_LENGTH
    for key in ("hy", "ay", "by"):
        tracking[key] = tracking[key] * PITCH_WIDTH
    print("koordinat dikonversi dari skala 0-1 ke meter")

del raw_df, dataset
gc.collect()

ball_nan = np.isnan(tracking["bx"]) | np.isnan(tracking["by"])
print("tim (metadata)      :", team_names)
print("slot home / away    :", len(slots["home"]), "/", len(slots["away"]))
print("bola NaN (semua)    : {:.1%}".format(ball_nan.mean()))
print("bola NaN (graded)   : {:.1%}".format(ball_nan[graded_mask].mean()))
print("rentang ball_x (m)  : {:.1f} .. {:.1f}".format(np.nanmin(tracking["bx"]), np.nanmax(tracking["bx"])))
print("ball_speed dari ATD :", tracking["bspeed"] is not None)

In [ ]:
events = load_smart_tagging.load()
print("instance Smart Tagging:", len(events))

code_counts = pd.Series([e["code"] for e in events]).value_counts()
print("\nkode:\n", code_counts.to_string())

tag_values = sorted({e["tags"].get("Team") for e in events} - {None})
print("\nnilai tag Team:", tag_values)
print("tipe SET PIECES:", pd.Series([e["tags"].get("Type") for e in events
                                    if code_of(e) == load_smart_tagging.KICKOFF_CODE]).value_counts().to_dict())

set_piece_len = pd.Series([e["end_frame"] - e["start_frame"] for e in events
                           if code_of(e) == load_smart_tagging.KICKOFF_CODE])
print("\ndurasi SET PIECES (frame):\n", set_piece_len.describe().round(1).to_string())

time_base = load_smart_tagging.check_time_base(events)
print("\ncek time base:", time_base)

## Section 2 — Feature Engineering (26 fitur)
Urutan sel: **2.1** fitur tracking (kelompok A, B, C, E) → **2.2** pemetaan tim Smart Tagging → **2.3** fitur Smart Tagging (kelompok D) dan penyusunan matriks fitur.

Aturan fallback saat bola NaN (`ball_visible = 0`):
- Posisi bola diteruskan maksimal 10 frame (*forward-fill*). Jika tetap kosong, `ball_x_m/ball_y_m` diisi titik tengah lapangan.
- `dist_*_min` jatuh ke jarak centroid tim ke bola (posisi bola hasil forward-fill). Jika masih kosong, diisi median jarak saat bola terlihat.
- `dist_ratio` dan `ball_zone_density_ratio` = 1,0; `n_*_near_ball` = 0; `ball_speed` = 0.
- Semua fitur rolling bersifat **backward-only**.

In [ ]:
def tracking_features(t):
    hx, hy, ax, ay = t["hx"], t["hy"], t["ax"], t["ay"]
    n = len(t["bx"])
    visible = ~(np.isnan(t["bx"]) | np.isnan(t["by"]))
    bx = np.where(visible, t["bx"], np.nan)
    by = np.where(visible, t["by"], np.nan)
    bx_f = pd.Series(bx).ffill(limit=BALL_NAN_FORWARD_FILL).to_numpy()
    by_f = pd.Series(by).ffill(limit=BALL_NAN_FORWARD_FILL).to_numpy()
    bx_last = pd.Series(bx).ffill().to_numpy()
    by_last = pd.Series(by).ffill().to_numpy()

    d_home = np.hypot(hx - bx[:, None], hy - by[:, None])
    d_away = np.hypot(ax - bx[:, None], ay - by[:, None])
    dmin_home = np.nanmin(d_home, axis=1)
    dmin_away = np.nanmin(d_away, axis=1)
    n_home_near = np.where(visible, (d_home <= BALL_PROXIMITY_RADIUS).sum(axis=1), 0)
    n_away_near = np.where(visible, (d_away <= BALL_PROXIMITY_RADIUS).sum(axis=1), 0)

    cx_home, cy_home = np.nanmean(hx, axis=1), np.nanmean(hy, axis=1)
    cx_away, cy_away = np.nanmean(ax, axis=1), np.nanmean(ay, axis=1)
    dc_home = np.hypot(cx_home - bx_f, cy_home - by_f)
    dc_away = np.hypot(cx_away - bx_f, cy_away - by_f)
    dc_home_last = np.hypot(cx_home - bx_last, cy_home - by_last)
    dc_away_last = np.hypot(cx_away - bx_last, cy_away - by_last)

    ok = visible & np.isfinite(dmin_home) & np.isfinite(dmin_away)
    med_home = np.nanmedian(dmin_home[visible])
    med_away = np.nanmedian(dmin_away[visible])

    dist_home_min = np.where(visible & np.isfinite(dmin_home), dmin_home, dc_home)
    dist_home_min = np.where(np.isfinite(dist_home_min), dist_home_min, med_home)
    dist_away_min = np.where(visible & np.isfinite(dmin_away), dmin_away, dc_away)
    dist_away_min = np.where(np.isfinite(dist_away_min), dist_away_min, med_away)
    dist_ratio = np.where(ok, dmin_home / (dmin_away + 1e-5), 1.0)

    has_pos = ~np.isnan(bx_f)
    ball_x_m = np.where(has_pos, bx_f, PITCH_LENGTH / 2)
    ball_y_m = np.where(has_pos, by_f, PITCH_WIDTH / 2)
    if t["bspeed"] is not None:
        speed = t["bspeed"]
    else:
        speed = np.hypot(np.diff(bx, prepend=np.nan), np.diff(by, prepend=np.nan)) * FPS

    def fill(a, fallback):
        return pd.Series(a).ffill().fillna(fallback).to_numpy()

    centroid_home_x = fill(cx_home, PITCH_LENGTH / 2)
    centroid_away_x = fill(cx_away, PITCH_LENGTH / 2)

    idx = np.arange(n)
    last_seen = np.maximum.accumulate(np.where(visible, idx, -1))
    frames_since_ball_seen = np.where(last_seen >= 0, idx - last_seen, idx + 1)

    feats = pd.DataFrame({
        "dist_home_min": dist_home_min,
        "dist_away_min": dist_away_min,
        "dist_ratio": dist_ratio,
        "n_home_near_ball": n_home_near,
        "n_away_near_ball": n_away_near,
        "ball_zone_density_ratio": np.where(visible, n_home_near / (n_away_near + 1.0), 1.0),
        "ball_x_m": ball_x_m,
        "ball_y_m": ball_y_m,
        "ball_in_home_half": np.where(has_pos, (bx_f < 52.5).astype(float), 0.5),
        "dist_ball_to_sideline": np.where(has_pos, np.minimum(by_f, PITCH_WIDTH - by_f), PITCH_WIDTH / 2),
        "dist_ball_to_endline": np.where(has_pos, np.minimum(bx_f, PITCH_LENGTH - bx_f), PITCH_LENGTH / 2),
        "ball_speed": np.nan_to_num(speed, nan=0.0),
        "centroid_home_x": centroid_home_x,
        "centroid_away_x": centroid_away_x,
        "centroid_diff_x": centroid_home_x - centroid_away_x,
        "spread_home": fill(np.nanstd(hx, axis=1), 0.0),
        "spread_away": fill(np.nanstd(ax, axis=1), 0.0),
        "dist_centroid_home_to_ball": np.where(np.isfinite(dc_home), dc_home, dist_home_min),
        "dist_centroid_away_to_ball": np.where(np.isfinite(dc_away), dc_away, dist_away_min),
        "frames_since_ball_seen": frames_since_ball_seen,
        "ball_visible": visible.astype(float),
        "dist_ratio_roll25": pd.Series(dist_ratio).rolling(window=25, min_periods=1).mean().to_numpy(),
    })

    aux = {
        "ball_visible": visible,
        "use_ball": ok,
        "nearest_home_ok": dmin_home <= dmin_away,
        "centroid_home_closer": np.where(np.isfinite(dc_home_last) & np.isfinite(dc_away_last),
                                         dc_home_last <= dc_away_last, True),
    }
    return feats, aux


feat_tracking, aux = tracking_features(tracking)
print(feat_tracking.shape)
print(feat_tracking.describe().T[["mean", "min", "max"]].round(2).to_string())

In [ ]:
def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def name_similarity(a, b):
    return SequenceMatcher(None, normalize_name(a), normalize_name(b)).ratio()


def map_by_name(values, home_name, away_name):
    if len(values) != 2:
        return None
    v0, v1 = values
    straight = name_similarity(v0, home_name) + name_similarity(v1, away_name)
    swapped = name_similarity(v0, away_name) + name_similarity(v1, home_name)
    if max(straight, swapped) < 1.2 or abs(straight - swapped) < 0.2:
        return None
    return {v0: 0, v1: 1} if straight > swapped else {v0: 1, v1: 0}


phase_events = [e for e in events
                if code_of(e) != load_smart_tagging.KICKOFF_CODE and e["tags"].get("Team")]


def home_share(value):
    mask = intervals_to_mask([(e["start_frame"], e["end_frame"])
                              for e in phase_events if e["tags"]["Team"] == value])
    mask &= aux["use_ball"] & graded_mask
    return float(aux["nearest_home_ok"][mask].mean()) if mask.any() else np.nan


shares = {v: home_share(v) for v in tag_values}
if len(tag_values) == 2 and not any(np.isnan(list(shares.values()))):
    ordered = sorted(tag_values, key=lambda v: -shares[v])
    map_by_tracking = {ordered[0]: 0, ordered[1]: 1}
else:
    map_by_tracking = {v: (0 if shares[v] > 0.5 else 1) for v in tag_values if not np.isnan(shares[v])}

map_by_names = map_by_name(tag_values, team_names.get("home", ""), team_names.get("away", ""))

if map_by_names is not None and map_by_names != map_by_tracking:
    print("PERINGATAN: pemetaan nama dan pemetaan posisi pemain berbeda. Dipakai pemetaan posisi pemain.")
    team_side = map_by_tracking
else:
    team_side = map_by_names or map_by_tracking

print("share pemain Home terdekat per tag Team:", {k: round(v, 3) for k, v in shares.items()})
print("pemetaan nama    :", map_by_names)
print("pemetaan tracking:", map_by_tracking)
print("pemetaan final (0=Home, 1=Away):", team_side)
assert set(team_side.values()) == {0, 1}, "Tag Team tidak terpetakan ke dua sisi berbeda."

In [ ]:
def smart_tagging_features(events, team_side):
    kick_code = load_smart_tagging.KICKOFF_CODE
    kick_type = load_smart_tagging.KICKOFF_TYPE
    kick_frames = load_smart_tagging.KICKOFF_VIDEO_FRAMES

    sp_all, sp_intervals, post_windows = [], [], []
    for e in events:
        if code_of(e) != kick_code:
            continue
        start, end = e["start_frame"], e["end_frame"]
        sp_all.append((start, end))
        if tag_of(e, "Type") == kick_type:
            kick = kick_frames.get(e["tags"].get("Half"))
            if kick is not None:
                end = min(end, kick)
        else:
            post_windows.append((e["end_frame"], e["end_frame"] + POST_SET_PIECE_DEAD_FRAMES))
        if end > start:
            sp_intervals.append((start, end))

    sp_mask = intervals_to_mask(sp_intervals)
    sp_mask_buf = intervals_to_mask([(s - SET_PIECE_BUFFER_FRAMES, e) for s, e in sp_intervals])
    goal_mask = intervals_to_mask([(e["start_frame"], e["end_frame"]) for e in events
                                   if code_of(e) in ("goals", "goal")])
    dead_phase = sp_mask | goal_mask | intervals_to_mask(post_windows)

    team_events = sorted((e for e in events if e["tags"].get("Team") in team_side),
                         key=lambda e: (e["start_frame"], -(e["end_frame"] - e["start_frame"])))
    smart_tag_team = np.full(N_FRAMES, -1, dtype=np.int8)
    for e in team_events:
        smart_tag_team[max(0, e["start_frame"]):min(N_FRAMES, e["end_frame"])] = team_side[e["tags"]["Team"]]

    since = np.full(N_FRAMES, 500, dtype=np.int64)
    idx = np.arange(N_FRAMES)
    for start, end in sp_all:
        lo, hi = max(0, end), min(N_FRAMES, end + 500)
        if hi > lo:
            since[lo:hi] = np.minimum(since[lo:hi], idx[lo:hi] - end)
    since[intervals_to_mask(sp_all)] = 0

    feats = pd.DataFrame({
        "is_set_piece": sp_mask.astype(float),
        "is_dead_ball_phase": dead_phase.astype(float),
        "smart_tag_team": smart_tag_team.astype(float),
        "frames_since_set_piece": np.clip(since, 0, 500).astype(float),
    })
    return feats, smart_tag_team, sp_mask, sp_mask_buf


feat_tagging, smart_tag_team, sp_mask, sp_mask_buf = smart_tagging_features(events, team_side)

feat = pd.concat([feat_tracking, feat_tagging], axis=1)[FEATURES].astype(np.float32)
assert feat.shape == (N_FRAMES, 26), feat.shape
assert not feat.isna().any().any(), feat.columns[feat.isna().any()].tolist()

print("matriks fitur:", feat.shape)
print("frame set piece (raw / +buffer):", int(sp_mask.sum()), "/", int(sp_mask_buf.sum()))
print("frame punya tag Team (graded)  : {:.1%}".format((smart_tag_team[graded_mask] >= 0).mean()))

## Section 3 — Weak Label Generation
Prioritas label (tinggi ke rendah): `SET PIECES` → D; `HALF2_PRE` → D; frame tidak graded → X (tidak dilatih); tag `Team` → H/A; sisanya heuristik pemain terdekat ke bola (bola hilang: tim dengan centroid lebih dekat ke posisi bola terakhir yang diketahui).

Weak label **bukan** ground truth. Sel ini juga mencetak seberapa sering tag `Team` cocok dengan heuristik pemain terdekat, sebagai gambaran kualitas noise.

In [ ]:
by_ball = np.where(aux["nearest_home_ok"], H_ID, A_ID)
by_centroid = np.where(aux["centroid_home_closer"], H_ID, A_ID)
heuristic = np.where(aux["use_ball"], by_ball, by_centroid)

weak = heuristic.astype(np.int8)
tag_ok = smart_tag_team >= 0
weak[tag_ok] = smart_tag_team[tag_ok]
weak[sp_mask] = D_ID
weak[HALF2_PRE[0]:HALF2_PRE[1] + 1] = D_ID
weak[~graded_mask] = X_ID

compare = tag_ok & aux["use_ball"] & graded_mask & ~sp_mask
print("distribusi weak label:", {str(ID_TO_LABEL[k]): int((weak == k).sum()) for k in (H_ID, A_ID, D_ID, X_ID)})
print("kecocokan tag Team vs pemain terdekat: {:.1%} dari {} frame".format(
    (smart_tag_team[compare] == heuristic[compare]).mean(), int(compare.sum())))

## Section 4 — LightGBM Emission Model
Model dilatih pada weak label (frame bukan X). Output yang dipakai adalah **probabilitas** 3 kelas, bukan label langsung. Public GT hanya dipakai untuk melihat log-loss, **tanpa** mengubah hyperparameter.

Catatan: `smart_tag_team` dan `is_set_piece` ikut menyusun weak label, sehingga model cenderung mengandalkan keduanya. Itu konsekuensi spesifikasi, bukan bug.

In [ ]:
LGBM_PARAMS = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "num_leaves": 31,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_samples": 50,
    "random_state": 42,
    "verbose": -1,
    "n_jobs": -1,
}

train_idx = np.flatnonzero(weak >= 0)
model = lgb.LGBMClassifier(**LGBM_PARAMS)
model.fit(feat.iloc[train_idx], weak[train_idx])
assert list(model.classes_) == [H_ID, A_ID, D_ID], model.classes_

emission = model.predict_proba(feat)
print("data latih:", len(train_idx), "frame | emission:", emission.shape)

In [ ]:
gt_path = glob.glob(str(ROOT / "evaluation" / "gt" / "*_possession_gt_public.csv"))[0]
gt = pd.read_csv(gt_path).set_index("frame").reindex(np.arange(N_FRAMES))
gt_ids = gt["label"].map(LABEL_TO_ID).fillna(X_ID).to_numpy().astype(int)
assert len(gt_ids) == N_FRAMES

val = gt_ids >= 0
print("frame GT publik berlabel:", int(val.sum()), "| rentang:", int(np.flatnonzero(val).min()), "-", int(np.flatnonzero(val).max()))
print("log-loss emission vs GT publik: {:.4f}  (acuan tebakan seragam = {:.4f})".format(
    log_loss(gt_ids[val], emission[val], labels=[H_ID, A_ID, D_ID]), np.log(3)))
print("macro-F1 argmax vs GT publik  : {:.4f}  (informatif saja)".format(
    f1_score(gt_ids[val], emission[val].argmax(axis=1), labels=[H_ID, A_ID, D_ID], average="macro")))

importance = pd.Series(model.booster_.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
top5 = importance.index[:5].tolist()
print("\ntop-5 fitur (gain):", top5)
missing = [f for f in ("dist_ratio", "dist_home_min", "is_set_piece") if f not in top5]
if missing:
    print("CATATAN: {} tidak masuk top-5. Cek apakah smart_tag_team mendominasi; periksa bug fitur bila fitur jarak nyaris tak berkontribusi.".format(missing))

ax = importance.head(15)[::-1].plot.barh(figsize=(8, 5), title="Feature importance (gain)")
plt.tight_layout()
plt.show()

## Section 5 — HMM + Viterbi Decoding
Matriks transisi dibuat manual (bukan dipelajari dari data) dengan state 0=H, 1=A, 2=D. Viterbi dijalankan **per babak** (tidak melewati halftime). Saat bola tidak terlihat, bobot emission turun dari 0,7 ke 0,3 sehingga inersia state sebelumnya lebih dominan.

Rekursi memakai *broadcasting* NumPy pada matriks 3x3 (tanpa loop bersarang); hanya ada satu loop waktu karena Viterbi memang berurutan.

Prior kickoff babak 1: state awal diberi peluang `KICKOFF_PRIOR` (0,80, asumsi saya) untuk tim yang melakukan kickoff menurut tag Smart Tagging; sisanya dibagi rata.

In [ ]:
TRANSITION_MATRIX = np.array([
    [TRANSITION_H_STAY, 0.0005, 1 - TRANSITION_H_STAY - 0.0005],
    [0.0005, TRANSITION_H_STAY, 1 - TRANSITION_H_STAY - 0.0005],
    [(1 - TRANSITION_D_STAY) / 2, (1 - TRANSITION_D_STAY) / 2, TRANSITION_D_STAY],
])
assert np.allclose(TRANSITION_MATRIX.sum(axis=1), 1.0)
INITIAL_PROB = np.array([0.33, 0.33, 0.34])


def viterbi_decode(emission_probs, trans_matrix, init_prob, emission_weight=None):
    n_frames, n_states = emission_probs.shape
    log_trans = np.log(trans_matrix + 1e-10)
    log_emit = np.log(emission_probs + 1e-10)
    if emission_weight is not None:
        log_emit = log_emit * emission_weight[:, None]

    score = np.empty((n_frames, n_states))
    back = np.zeros((n_frames, n_states), dtype=np.int64)
    cols = np.arange(n_states)
    score[0] = np.log(init_prob + 1e-10) + log_emit[0]

    for t in range(1, n_frames):
        candidates = score[t - 1][:, None] + log_trans
        back[t] = candidates.argmax(axis=0)
        score[t] = candidates[back[t], cols] + log_emit[t]

    states = np.empty(n_frames, dtype=np.int64)
    states[-1] = score[-1].argmax()
    for t in range(n_frames - 2, -1, -1):
        states[t] = back[t + 1, states[t + 1]]
    return states


def kickoff_side(half_label):
    for e in events:
        if (code_of(e) == load_smart_tagging.KICKOFF_CODE
                and tag_of(e, "Type") == load_smart_tagging.KICKOFF_TYPE
                and e["tags"].get("Half") == half_label):
            return team_side.get(e["tags"].get("Team"))
    return None


def prior_with_kickoff(side):
    if side is None:
        return INITIAL_PROB
    prior = np.full(3, (1 - KICKOFF_PRIOR) / 2)
    prior[side] = KICKOFF_PRIOR
    return prior


emission_weight = np.where(feat["ball_visible"].to_numpy() == 1.0, EMISSION_WEIGHT_BALL, EMISSION_WEIGHT_NO_BALL)
first_half_side = kickoff_side("1st Half")
print("tim kickoff babak 1 (0=Home, 1=Away, None=tidak ada):", first_half_side)

viterbi_ids = np.full(N_FRAMES, X_ID, dtype=np.int64)
for (lo, hi), init in ((HALF1, prior_with_kickoff(first_half_side)), (HALF2, INITIAL_PROB)):
    sl = slice(lo, hi + 1)
    viterbi_ids[sl] = viterbi_decode(emission[sl], TRANSITION_MATRIX, init, emission_weight[sl])


def count_segments(ids, block):
    seg = ids[block[0]:block[1] + 1]
    return int((np.diff(seg) != 0).sum() + 1)


argmax_ids = emission.argmax(axis=1)
for name, block in (("babak 1", HALF1), ("babak 2", HALF2)):
    print("{}: segmen argmax = {} -> segmen Viterbi = {}".format(
        name, count_segments(argmax_ids, block), count_segments(viterbi_ids, block)))

## Section 6 — Hard Post-Processing Rules
Urutan aplikasi: (1) frame non-graded → X, (2) pra-kickoff babak 2 → D, (3) interval `SET PIECES` + buffer 5 frame → D, (4) *pruning* segmen H/A yang lebih pendek dari 15 frame (digabung ke tetangga yang lebih panjang), (5) abstain `X` untuk bola hilang lebih dari 75 frame berturut-turut **hanya jika** label sebelum, selama, dan sesudah celah sama (bukan titik transisi). Karena evaluator melakukan forward-fill, aturan 5 tidak mengubah hasil label; ia bisa dimatikan lewat `APPLY_ABSTAIN_RULE = False`.

In [ ]:
def prune_short_spells(ids, block, min_len=MIN_SPELL_FRAMES, max_pass=50):
    lo, hi = block
    seg = ids[lo:hi + 1].copy()
    for _ in range(max_pass):
        change = np.flatnonzero(np.diff(seg) != 0) + 1
        starts = np.concatenate([[0], change])
        ends = np.concatenate([change, [len(seg)]])
        lens = ends - starts
        vals = seg[starts]
        short = np.flatnonzero((lens < min_len) & (vals != D_ID) & (vals != X_ID))
        if len(short) == 0:
            break
        for i in short:
            left = lens[i - 1] if i > 0 else -1
            right = lens[i + 1] if i < len(lens) - 1 else -1
            if left < 0 and right < 0:
                continue
            seg[starts[i]:ends[i]] = vals[i - 1] if left >= right else vals[i + 1]
    out = ids.copy()
    out[lo:hi + 1] = seg
    return out


def abstain_long_ball_gaps(ids, ball_visible, blocks, locked_mask, min_gap=ABSTAIN_MIN_BALL_GAP):
    out = ids.copy()
    n_applied = 0
    for lo, hi in blocks:
        vis = ball_visible[lo:hi + 1].astype(np.int8)
        change = np.flatnonzero(np.diff(vis) != 0) + 1
        starts = np.concatenate([[0], change])
        ends = np.concatenate([change, [len(vis)]])
        for a, b in zip(starts, ends):
            if vis[a] == 1 or (b - a) <= min_gap:
                continue
            ga, gb = lo + a, lo + b
            if ga == lo or gb > hi or locked_mask[ga:gb].any():
                continue
            gap = ids[ga:gb]
            label = gap[0]
            if label not in (H_ID, A_ID) or (gap != label).any():
                continue
            if ids[ga - 1] != label or ids[gb] != label:
                continue
            out[ga:gb] = X_ID
            n_applied += 1
    return out, n_applied


final_ids = viterbi_ids.copy()

for lo, hi in (PRE_MATCH, HALFTIME, POST_MATCH):
    final_ids[lo:hi + 1] = X_ID
final_ids[HALF2_PRE[0]:HALF2_PRE[1] + 1] = D_ID
final_ids[sp_mask_buf & graded_mask] = D_ID

for block in (HALF1, HALF2):
    final_ids = prune_short_spells(final_ids, block)

n_abstain = 0
if APPLY_ABSTAIN_RULE:
    final_ids, n_abstain = abstain_long_ball_gaps(final_ids, aux["ball_visible"], (HALF1, HALF2), sp_mask_buf)

print("celah bola >{} frame yang di-abstain: {}".format(ABSTAIN_MIN_BALL_GAP, n_abstain))
print("jumlah frame berubah oleh post-processing:", int((final_ids != viterbi_ids).sum()))
print("distribusi akhir:", {str(ID_TO_LABEL[k]): int((final_ids == k).sum()) for k in (H_ID, A_ID, D_ID, X_ID)})

## Section 7 — Submission & Evaluation
CSV berisi 199.319 baris (`frame`, `label`). Setelah ditulis ke `evaluation/results/our_submission.csv`, sel berikut menjalankan `validate_submission.py` dan `run_evaluation.py` (set `BOOTSTRAP_RUNS = 0` di sel 0.2 untuk melewati bootstrap).

Target sanity check: 150–250 possession spell, rata-rata durasi 10–20 detik, spellF1 publik > 0,35 (lantai baseline 0,2433), macroF1 publik > 0,60.

In [ ]:
labels = ID_TO_LABEL[final_ids]
submission = pd.DataFrame({"frame": np.arange(N_FRAMES), "label": labels})
assert len(submission) == N_FRAMES
assert set(submission["label"].unique()) <= {"H", "A", "D", "X"}

submission_path = RESULTS_DIR / "our_submission.csv"
submission.to_csv(submission_path, index=False)
print("tersimpan:", submission_path)
print(submission["label"].value_counts().to_string())


def simple_spell_stats(label_array):
    filled = pd.Series(np.where(label_array == "X", None, label_array)).ffill().to_numpy()
    alive = np.isin(filled, ["H", "A"])
    starts = alive & np.concatenate([[True], (filled[1:] != filled[:-1]) | ~alive[:-1]])
    n_spells = int(starts.sum())
    mean_s = alive.sum() / FPS / n_spells if n_spells else 0.0
    return n_spells, mean_s


n_spells, mean_s = simple_spell_stats(labels)
print("\nperkiraan jumlah spell: {} (target 150-250) | rata-rata durasi: {:.1f} s (target 10-20 s)".format(n_spells, mean_s))

In [ ]:
check = subprocess.run([sys.executable, "evaluation/validate_submission.py", str(submission_path)],
                       capture_output=True, text=True, cwd=ROOT)
print(check.stdout)
print(check.stderr)
assert check.returncode == 0, "Validasi submission gagal."

In [ ]:
cmd = [sys.executable, "evaluation/run_evaluation.py"]
if BOOTSTRAP_RUNS:
    cmd += ["--bootstrap", str(BOOTSTRAP_RUNS)]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
print(result.stdout)
print(result.stderr)

## Section 8 — Error Analysis (opsional)
Hanya untuk membaca perilaku model pada 5 menit GT publik. Jangan dipakai untuk mengubah parameter.

In [ ]:
lo, hi = PUBLIC_VAL_RANGE
sl = slice(lo, hi + 1)
mask = gt_ids[sl] >= 0
print(pd.crosstab(pd.Series(ID_TO_LABEL[gt_ids[sl]][mask], name="GT"),
                  pd.Series(labels[sl][mask], name="Prediksi")).to_string())

cmap = ListedColormap(["#bbbbbb", "#d62728", "#1f77b4", "#2ca02c"])
fig, axes = plt.subplots(2, 1, figsize=(14, 2.6), sharex=True)
for ax, ids, title in ((axes[0], gt_ids[sl], "GT publik"), (axes[1], final_ids[sl], "Prediksi")):
    ax.imshow(ids[None, :] + 1, aspect="auto", cmap=cmap, vmin=0, vmax=3, interpolation="nearest")
    ax.set_yticks([])
    ax.set_ylabel(title, rotation=0, ha="right", va="center")
axes[1].set_xlabel("frame sejak {} (abu=X, merah=H, biru=A, hijau=D)".format(lo))
plt.tight_layout()
plt.show()

print("\nspell GT publik: {} | spell prediksi pada rentang yang sama: {}".format(
    simple_spell_stats(ID_TO_LABEL[gt_ids[sl]])[0], simple_spell_stats(labels[sl])[0]))